In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')
_os.environ.setdefault('AWS_REGION', 'us-west-2')
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


## RLAIF Example - Finetuning with Sagemaker

This notebook demonstrates basic user flow for RLAIF Finetuning from a model available in Sagemaker Jumpstart.
Information on available models on jumpstart: https://docs.aws.amazon.com/sagemaker/latest/dg/jumpstart-foundation-models-latest.html

### Setup and Configuration

Initialize the environment by importing necessary libraries and configuring AWS credentials

In [2]:
# Configure AWS credentials and region
#! ada credentials update --provider=isengard --account=<> --role=Admin --profile=default --once
#! aws configure set region us-west-2

In [3]:
#!/usr/bin/env python3

from sagemaker.train.rlaif_trainer import RLAIFTrainer
from sagemaker.train.configs import InputData
from rich import print as rprint
from rich.pretty import pprint
from sagemaker.core.resources import ModelPackage
import os
#os.environ['SAGEMAKER_REGION'] = 'us-east-1'

import boto3
from sagemaker.core.helper.session_helper import Session

# For MLFlow native metrics in Trainer wait, run below line with approriate region
os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = "https://mlflow.sagemaker.us-west-2.app.aws"



sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


### Prepare and Register Dataset
Prepare and Register Dataset for Finetuning

In [4]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique



# Register dataset in SageMaker AI Registry
# This creates a versioned dataset that can be referenced by ARN
# Provide a source (it can be local file path or S3 URL)
dataset = DataSet.create(
    name="demo-2",
    source="s3://notebook-test-engine-ds-674622101542-usw2/datasets/sample-converse-messages.jsonl"
)

print(f"Dataset ARN: {dataset.arn}")

[07/17/26 00:35:18] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=2790472;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=2790473;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/17/26 00:35:20] INFO     Cannot simulate policies for                                  ]8;id=2790480;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=2790481;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=2790487;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=2790488;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Role not provided. Using validated caller role:                         ]8;id=2790495;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=2790496;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#90\90]8;;\
                             arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole                   

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=2790501;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=2790502;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

...

[07/17/26 00:35:24] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=2790525;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=2790526;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/usr/local/lib/python3.12/dist-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

[07/17/26 00:35:25] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=2790531;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=2790532;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

Final Resource Status: Available

Dataset ARN: arn:aws:sagemaker:us-west-2:674622101542:hub-content/R8VBAD7SJ9M4M5ATDE0MIPSEJU3M1TEH5UCUNB65JMGNEP87RHFG/DataSet/demo-2/9.0.0


##### Create a Model Package group (if not already exists)

In [5]:
from sagemaker.core.resources import ModelPackage, ModelPackageGroup

model_package_group=ModelPackageGroup.create(model_package_group_name="test-model-package-group")

                    INFO     Creating model_package_group resource.                              ]8;id=2790539;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=2790540;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#23250\23250]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=2790547;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=2790548;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#361\361]8;;\

                    INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=2790554;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=2790555;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#375\375]8;;\

### Create RLAIFTrainer
**Required Parameters** 

* `model`: base_model id on Sagemaker Hubcontent that is available to finetune (or) ModelPackage artifacts

**Optional Parameters**
* `reward_model_id`: Bedrock model id to be used as judge.
*  `reward_prompt`: Reward prompt ARN or builtin prompts refer: https://docs.aws.amazon.com/bedrock/latest/userguide/model-evaluation-metrics.html
* `model_package_group`: ModelPackage group name or ModelPackageGroup object. This parameter is mandatory when a base model ID is provided, but optional when a model package is provided.
* `mlflow_resource_arn`: MLFlow app ARN to track the training job
* `mlflow_experiment_name`: MLFlow app experiment name(str)
* `mlflow_run_name`: MLFlow app run name(str)
* `training_dataset`: Training Dataset - should be a Dataset ARN or Dataset object (Please note training dataset is required for a training job to run, can be either provided via Trainer or .train())
* `validation_dataset`: Validation Dataset - should be a Dataset ARN or Dataset object
* `s3_output_path`: S3 path for the trained model artifacts

#### Reference 
Refer this doc for other models that support Model Customization: 
https://docs.aws.amazon.com/bedrock/latest/userguide/custom-model-supported.html

Refer this for supported reward models: 
https://github.com/aws/sagemaker-python-sdk/blob/master/sagemaker-train/src/sagemaker/train/constants.py#L46

In [6]:
# For fine-tuning 
rlaif_trainer = RLAIFTrainer(
    model="meta-textgeneration-llama-3-2-1b-instruct", 
    model_package_group=model_package_group, # or use an existing model package group arn
    reward_model_id='openai.gpt-oss-120b-1:0',
    reward_prompt='Builtin.Summarize',
    mlflow_experiment_name="test-rlaif-finetuned-models-exp", 
    mlflow_run_name="test-rlaif-finetuned-models-run", 
    training_dataset=dataset.arn, 
    s3_output_path="s3://notebook-test-engine-ds-674622101542-usw2/output/",
    accept_eula=True
)

[07/17/26 00:35:26] INFO     SageMaker session not provided. Using default Session.                  ]8;id=2790561;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=2790562;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=2790567;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=2790568;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

### Discover and update Finetuning options

Each of the technique and model has overridable hyperparameters that can be finetuned by the user.

In [7]:
print("Default Finetuning Options:")
pprint(rlaif_trainer.hyperparameters.to_dict()) # rename as hyperparameters

#set options
rlaif_trainer.hyperparameters.get_info()


Default Finetuning Options:


{
│   'clip_ratio': '0.2',
│   'clip_ratio_high': '0.2',
│   'clip_ratio_low': '0.2',
│   'global_batch_size': '128',
│   'judge_model_id': 'bedrock/openai.gpt-oss-120b-1:0',
│   'judge_prompt_template': '/opt/ml/code/verl/summarize.jinja',
│   'kl_loss_coef': '0.001',
│   'learning_rate': '1e-05',
│   'lora_alpha': '64',
│   'lora_rank': '32',
│   'lr_warmup_steps_ratio': '0.0',
│   'max_epochs': '2',
│   'max_prompt_length': '4000',
│   'min_lr': '0.0',
│   'mlflow_run_id': '',
│   'mlflow_tracking_uri': '',
│   'model_name_or_path': 'meta-llama/Llama-3.2-1B-Instruct',
│   'name': 'example-name-b1n03',
│   'results_directory': '',
│   'resume_from_path': '',
│   'rollout_n': '8',
│   'rollout_temperature': '1.0',
│   'temperature': '0',
│   'train_val_split_ratio': '0.9',
│   'use_kl_loss': 'True',
│   'warmup_steps': '-1',
│   'weight_decay': '0.01'
}

[07/17/26 00:35:27] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=2790573;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=2790574;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             


clip_ratio:
  Current value: 0.2
  Type: float
  Default: 0.2
  Range: 0.1 - 1.5
  Required: Yes

clip_ratio_high:
  Current value: 0.2
  Type: float
  Default: 0.2
  Range: 0.0 - 0.5
  Required: Yes

clip_ratio_low:
  Current value: 0.2
  Type: float
  Default: 0.2
  Range: 0.0 - 0.5
  Required: Yes

global_batch_size:
  Current value: 128
  Type: integer
  Default: 128
  Valid options: [128, 256, 512, 1024]
  Required: Yes

judge_model_id:
  Current value: bedrock/openai.gpt-oss-120b-1:0
  Type: string
  Default: bedrock/openai.gpt-oss-120b-1:0
  Required: Yes

judge_prompt_template:
  Current value: /opt/ml/code/verl/summarize.jinja
  Type: string
  Default: /opt/ml/code/verl/summarize.jinja
  Valid options: ['/opt/ml/code/verl/cot.jinja', '/opt/ml/code/verl/evaluate.jinja', '/opt/ml/code/verl/faithfulness.jinja', '/opt/ml/code/verl/summarize.jinja', 'bedrock/RLAIF/PandaLM/prompts/grader.jinja']

kl_loss_coef:
  Current value: 0.001
  Type: float
  Default: 0.001
  Range: 0 - 0.1
 

#### Start RLAIF training


In [8]:

training_job = rlaif_trainer.train(wait=True)


╭────────────────────────────────── Training Job Status ───────────────────────────────────╮
│  TrainingJob Name      meta-textgeneration-llama-3-2-1b-instruct-rlaif-20260717003529    │
│  TrainingJob ARN       arn:aws:sagemaker:us-west-2:674622101542:training-job/meta-textg  │
│                        eneration-llama-3-2-1b-instruct-rlaif-20260717003529              │
│  MLflow Experiment     test-rlaif-finetuned-models-exp                                   │
│  Links                 ]8;id=2795912;https://us-west-2.console.aws.amazon.com/sagemaker/home?region=us-west-2#/jobs/meta-textgeneration-llama-3-2-1b-instruct-rlaif-20260717003529\🔗 Training Job (Console)]8;;\                                         │
│                        ]8;id=2795917;https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs$3FlogStreamNameFilter$3Dmeta-textgeneration-llama-3-2-1b-instruct-rlaif-20260717003529\🔗 CloudWatch Logs]8;;\ | ]8;id=2795918;https://app-3THB6MFIDE4F.mlflow.sagemaker.us-west-2.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6Ilc3TUtJUiIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNDJPbThrVEVrb1d3Nm1vSzZDYjBzMWZwWURtRmNaU3B2bVBqMWFqSlZoYk1BWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGdllVVkJkVEpSYmtNNVNFZEZXRlpuZFRaVFVuSkNRVU01YmpSbWVsRTFTbVZ1V0hOUGMzQnlPRmRXWW14VVJ6bEZlbVZGT0U5cmJTc3JjVk56ZWtoSFVUMDlBQUVBQjJGM2N5MXJiWE1BUzJGeWJqcGhkM002YTIxek9uVnpMWGRsYzNRdE1qbzVOell4T1RNeU5UUTFOemM2YTJWNUwyVmhNbUZrTUdSa0xXRmtNV0l0TkROalppMDRaak0xTFRNMU5UWmlZVE5pWm1NeE1nQzRBUUlCQUhqRHJwaCtQMVNKcHUzZUl2YnVXUi94bGQ2THMxNjhEU3JZbXo5akp0dlBUQUZ5WnowVUQvM3VwYk5lQ3YvL2VubWRBQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU1FV016UHZKN1dvRm5GV1pUQWdFUWdEdWZHYTB1MysyajBKWEFCR21zbnIybUJnNkVrbEp0ZmM4RldlM2M4NkcwbVk1QXEzMG9iYjlEdmRWSytMSVFQejVSVVdPYytyODYxbDhNSmdJQUFCQUE3R3VmbjZCb0JPV2hOMWtzME5BYjNZSG9BZ2x2T1pjQzJsTlhET2c5TnBMOERRWGxHOEpRZVpKOUlrdDMyUFczLy8vLy93QUFBQUVBQUFBQUFBQUFBQUFBQUFFQUFBUVhIT0NwdDZTVWZxc2hzNmltQWV4L2tLb0NQU3BVdFlrWUhnTXlHckh2RFpMN1dwcnp4a1VRM05QcE1GYXlrQmRQSDVvQlpJS21vTFEyMStNUmQ5TzJHYWtHaVhqSGt3M3FlVEtyN0NYNGI1RVFhSks1NEdydXJobzdyM2V5aVMzUWVtSjNhazBVaXoxRHkvcjJwcFBGd21PWjFNZXRreU1hNy8xZitnQWQrd3BmYnZBZU9iOU1LSlJXSlJmODRjWjJyd3VKQ0NGNjBDZ0Q3WCtwb1c0ZUVrM1poc3lmVkF4M3VNTy90UU9xb2UranNSVmxxVmVIQnJOYVRySUlwZnA4OVIyQ21BVUF4TkdVMC9KcmwwK0xucSt6bERxZ1hBSDZYVTNJMzBnZFdjRTJFaDlNL0FaQXByWC8yY0d6RWdOVVRyd2dKTXBhRE85SG4xUkZSVmZYUnFFdlMvM3oveHNmVks2WSt0VUVrYzVkakZ4SW4ydDVvRDloUjU3OWNobnBJOG1IQ1hiSkNZeG01UkxLcjJvNXl6WnI4MXUxUzl6M2FPSzRVQnF0NDI2NWFOV3ZicitvbVhmcURmalBMS01VOXYyTVdGT0pVMHV4VitrSlAwZ0hZNjkrR1FxU2VZc2tIdXl3NG9FbnNjbFpORmNETFpDd2hxRTdJQWtRWTBlSm5mb0JKZkRDSnJyVG9uQ0lGSU9yNEhGYkFZbDFtMkowWkJBM2dETUFIUlVtL1pjTE1Odk1pWTJMcTk3VWhVWnlkWlhLdlJsYUdRaHJmSEtxbWJ6SXpOSldYOGlhN3Awb05Ta2xaVDEyQnJqamhxZ2NDQkFFcXNNQ1hrV3RRRUxYSjNBdWJQL1ZsS20rL1FqQjN1NlluaGllNlR0bHpmUEttNHBPdURGaGo4Z3ZwTlNhVVpZREovb1JkRGM4S0VTMFRncUpHRzFtZmc0U0JINXcvL0ZWbGdJZjZGMU40YnFucmNoalJDVnJ6eFhxeURERUZ0NHVIcWRiVWVYMkNuMXNGTzFOY01TZVNNb3B6QWJyS0cvdC9tZGxNT3AybVhKMFl6UlUyR0kycEMxaXhBRG56MTZLWmRLU1hZeXExMVNqeFo5ZStmcEMxdVZQai8xNTdQa3ZHeVdLWkR1UXpqdG5vWTcreVEyTVIweWtCNk55ZFRjbFZiVGlvY0FxVmUzaUphUFE0dWZGV29jM1k0T2pIWlVsdTAxV25WZCtBU3BBalU1SEQ1MVpEemtwWGNpTFFwSDQwczhWRFo3UnhEakJESDVDODZjZ3FHSjcvU0wrMGZVRjhib3czT05IaFA1b0dWeWNmT2lkdWRJSWNoekUySHZKbTBkdEp2VlM2ZU5Gc0hRNVZtTktXNGZJSVUwU04rdkJhdUozeWJLbDNaMC9EeTBicFJqQ2ZORFhSbWJid0JsbXdmbmRZVVVKMjFKeTZJK2IvTEhzamZwU2RRSk16Y2FrNVd5UnJ3UHM3RTgvaElqSm53TnR2ZFZrTnpNWjVRdW9aZ09uS3pxdUMyblBQb1YxTVBEQVJNUzhwM0Z4OUJrRXZmT01VOE5IMWs3YnJNTmxmM2dFMmc3bEhvcndXUzJMMkJhdjZ2UkRpcUcrUmhFSE9nMnpnamI5K2U2aUdsMmxBeWNqYmp1OU1WRnVicWFVNTF0bGRBRXY0d2szbWhYQzlydjZTMnFEazhFU05URkoxSHhiaUdyVnVteVlwWmhBK1BkNFdxUXFVTEFhdTRueWYzWkdaV0JrTkhhYVNPcUdGbkdrUFU4d01uWWxPMGpzazdnSXBPNHdmZWhTM1JPeEZ3VnZzODByQjFPSkcyUW1wS3JJNXpaQUoxRUVVSjFQNUJ2eDg1WnE3SktWK0FnL2VOS2xxZ0JuTUdVQ01EcEV5dnJWSDl1bFM1VlpDMDMxWWtmNXRHQ0RlaW45TjAwNlRNcm9JTl

In [9]:
import re
from sagemaker.core.utils.utils import Unassigned
import json


def pretty_print(obj):
    def parse_unassigned(item):
        if isinstance(item, Unassigned):
            return None
        if isinstance(item, dict):
            return {k: parse_unassigned(v) for k, v in item.items() if parse_unassigned(v) is not None}
        if isinstance(item, list):
            return [parse_unassigned(x) for x in item if parse_unassigned(x) is not None]
        if isinstance(item, str) and "Unassigned object" in item:
            pairs = re.findall(r"(\w+)=([^<][^=]*?)(?=\s+\w+=|$)", item)
            result = {k: v.strip("'\"") for k, v in pairs}
            return result if result else None
        return item

    cleaned = parse_unassigned(obj.__dict__ if hasattr(obj, '__dict__') else obj)
    print(json.dumps(cleaned, indent=2, default=str))

pretty_print(training_job)

{
  "training_job_name": "meta-textgeneration-llama-3-2-1b-instruct-rlaif-20260717003529",
  "training_job_arn": "arn:aws:sagemaker:us-west-2:674622101542:training-job/meta-textgeneration-llama-3-2-1b-instruct-rlaif-20260717003529",
  "model_artifacts": "s3_model_artifacts='s3://notebook-test-engine-ds-674622101542-usw2/output/meta-textgeneration-llama-3-2-1b-instruct-rlaif-20260717003529/output/model'",
  "training_job_status": "Completed",
  "secondary_status": "Completed",
  "hyper_parameters": {
    "clip_ratio": "0.2",
    "clip_ratio_high": "0.2",
    "clip_ratio_low": "0.2",
    "global_batch_size": "128",
    "judge_model_id": "bedrock/openai.gpt-oss-120b-1:0",
    "judge_prompt_template": "/opt/ml/code/verl/summarize.jinja",
    "kl_loss_coef": "0.001",
    "learning_rate": "1e-05",
    "lora_alpha": "64",
    "lora_rank": "32",
    "lr_warmup_steps_ratio": "0.0",
    "max_epochs": "2",
    "max_prompt_length": "4000",
    "min_lr": "0.0",
    "model_name_or_path": "meta-llama

### View any Training job details

We can get any training job details and its status with TrainingJob.get(...)

In [10]:
from sagemaker.core.resources import TrainingJob

response = TrainingJob.get(training_job_name=training_job.training_job_name)
pretty_print(response)

{
  "training_job_name": "meta-textgeneration-llama-3-2-1b-instruct-rlaif-20260717003529",
  "training_job_arn": "arn:aws:sagemaker:us-west-2:674622101542:training-job/meta-textgeneration-llama-3-2-1b-instruct-rlaif-20260717003529",
  "model_artifacts": "s3_model_artifacts='s3://notebook-test-engine-ds-674622101542-usw2/output/meta-textgeneration-llama-3-2-1b-instruct-rlaif-20260717003529/output/model'",
  "training_job_status": "Completed",
  "secondary_status": "Completed",
  "hyper_parameters": {
    "clip_ratio": "0.2",
    "clip_ratio_high": "0.2",
    "clip_ratio_low": "0.2",
    "global_batch_size": "128",
    "judge_model_id": "bedrock/openai.gpt-oss-120b-1:0",
    "judge_prompt_template": "/opt/ml/code/verl/summarize.jinja",
    "kl_loss_coef": "0.001",
    "learning_rate": "1e-05",
    "lora_alpha": "64",
    "lora_rank": "32",
    "lr_warmup_steps_ratio": "0.0",
    "max_epochs": "2",
    "max_prompt_length": "4000",
    "min_lr": "0.0",
    "model_name_or_path": "meta-llama

### Test RLAIF with Custom Reward Prompt

Here we are providing a user-defined reward prompt/evaluator ARN

#### Create a custom reward prompt 

In [11]:
from rich.pretty import pprint

from sagemaker.ai_registry.air_constants import REWARD_FUNCTION, REWARD_PROMPT
from sagemaker.ai_registry.evaluator import Evaluator

evaluator = Evaluator.create(
    name = "jamj-rp2",
    source="s3://notebook-test-engine-ds-674622101542-usw2/fixtures/reward_prompt.jinja",
    type = REWARD_PROMPT
)

[07/17/26 00:52:43] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=2795923;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=2795924;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/17/26 00:52:44] INFO     Cannot simulate policies for                                  ]8;id=2795929;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=2795930;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=2795935;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=2795936;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Role not provided. Using validated caller role:                         ]8;id=2795941;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=2795942;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#90\90]8;;\
                             arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole                   

[07/17/26 00:52:45] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=2795947;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=2795948;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

...

[07/17/26 00:52:47] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=2795959;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=2795960;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=2795965;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=2795966;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/usr/local/lib/python3.12/dist-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

[07/17/26 00:52:48] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=2795971;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=2795972;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

Final Resource Status: Available

#### Use it with RLAIF Trainer

In [12]:


# For fine-tuning 
rlaif_trainer = RLAIFTrainer(
    model="meta-textgeneration-llama-3-2-1b-instruct", 
    model_package_group="sdk-test-finetuned-models",
    reward_model_id='openai.gpt-oss-120b-1:0',
    reward_prompt=evaluator.arn,
    mlflow_experiment_name="test-rlaif-finetuned-models-exp", 
    mlflow_run_name="test-rlaif-finetuned-models-run", 
    training_dataset=dataset.arn, 
    s3_output_path="s3://notebook-test-engine-ds-674622101542-usw2/output/",
    accept_eula=True
)


[07/17/26 00:52:49] INFO     SageMaker session not provided. Using default Session.                  ]8;id=2795977;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=2795978;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

[07/17/26 00:52:50] INFO     SageMaker session not provided. Using default Session.                  ]8;id=2795983;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=2795984;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

In [13]:
training_job = rlaif_trainer.train(wait=True)

╭────────────────────────────────── Training Job Status ───────────────────────────────────╮
│  TrainingJob Name      meta-textgeneration-llama-3-2-1b-instruct-rlaif-20260717005252    │
│  TrainingJob ARN       arn:aws:sagemaker:us-west-2:674622101542:training-job/meta-textg  │
│                        eneration-llama-3-2-1b-instruct-rlaif-20260717005252              │
│  MLflow Experiment     test-rlaif-finetuned-models-exp                                   │
│  Links                 ]8;id=2798959;https://us-west-2.console.aws.amazon.com/sagemaker/home?region=us-west-2#/jobs/meta-textgeneration-llama-3-2-1b-instruct-rlaif-20260717005252\🔗 Training Job (Console)]8;;\                                         │
│                        ]8;id=2798964;https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs$3FlogStreamNameFilter$3Dmeta-textgeneration-llama-3-2-1b-instruct-rlaif-20260717005252\🔗 CloudWatch Logs]8;;\ | ]8;id=2798965;https://app-3THB6MFIDE4F.mlflow.sagemaker.us-west-2.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IlQyVlhLNCIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNFViaFVuZXI5VXY1eGZuRHZvTnBxM0hYM05PcXBrYS84akdqQU1MaFJiQ29BWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGNGNEWXpTazVqYlRBdlVFMTNWUzgxTjNFdlJWaGhVV3hCWnl0R1JWTktTVzlCWjJSbWVteExSVWx4ZFN0VlpVbFJhRWxsTjBvelYweGxWR3hRYW1WWlFUMDlBQUVBQjJGM2N5MXJiWE1BUzJGeWJqcGhkM002YTIxek9uVnpMWGRsYzNRdE1qbzVOell4T1RNeU5UUTFOemM2YTJWNUwyVmhNbUZrTUdSa0xXRmtNV0l0TkROalppMDRaak0xTFRNMU5UWmlZVE5pWm1NeE1nQzRBUUlCQUhqRHJwaCtQMVNKcHUzZUl2YnVXUi94bGQ2THMxNjhEU3JZbXo5akp0dlBUQUZjcmZVRFR0eVdkYVRhcE92VXR4aXZBQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU1jVE9tUFdFMWFoZm4zMld3QWdFUWdEdGppSEJYN1NUckZ3eFJweFhSWnM4Z04rUjdGdlRFcFk1dWhCdHh3WHJOWStPU2hFa1E5L20rWnI1OS83Wi9sUTBiK0lUTHd1WUJ6ZFVGL3dJQUFCQUFEZ1BpUERoenVEY0dvckQrVlZDNVFaclNiN3lSeGxTeXlwaHdoN05jeGw3MWF6eGEyVGtUa3JLSEhOd2Nudm9mLy8vLy93QUFBQUVBQUFBQUFBQUFBQUFBQUFFQUFBUVh5dWZrdGk5a3B2RHgwa2RqTC82VmxJVkZYMldyZGdSMi9qYUxiUGhvYlptZTJ4bjRZZk5aM1oxb0FSbjRraEYzb21SWkdFMUJPVHZUcVJJUWN5Zml1U2VGZEdBMW9GQ1h0bWZIVkkyamJuSXVQMVpHVE1zUVRkRy83T1U1Z3YrZHZHYzhxbVF2aTFJVnVaZlJQbDlFN2RTWkpxc0U4dm1INHpadEkrWUFzRkRLWm1jK2dPVG5QNGpra0FGQ1h3ZXcwcFF2TUc5aURrOVVyQ3UyZC9wekdyejRMM1RZVGtjRHJDQzFLQWJ1amY2bTNHRHR0OEFTZENtT015cWUrUjFyK3E1ZTJ5NTZoRFNLVWNLbCtFMStRMmZMQVJ4MDl5S2IxWUR4UXFBQmdCNUtoQU9OTGs2d0tBR3ZzREh2MXpZL0YwK2hlNDQzdElmeDVWTE5zdjZwTWNFcktMVU1DWnFaY0VNd0NmMzRDTHFSLzBUZU1hbTN5ckRQbVNMVDVQWmVoTTZnOWE4VDRpVWowbElFbDJydElYQ0xMT2NtQ29xZEdDd0xZL3NPOGxoNUxIUlhMRnpyejhmbTJFeC9OamZqRVhyQnJ2WnVFRzRBc0s0bTY2ODJPVkxneGN2R1lrQjIvSXJMNVIySmFBYzZYUjlmd0FBRlhIYWZDVWFTWXBtd2ZOQ3hjVVRUWnBjbVA1RTA5bWdKTm5TNnJnU2pGaU5UZmczb2FudkdaNXE3K0ZWZk9veUhMbnNmR1UrdlBQNm52bUJETzhTdWczOXp1U2dOSC93TExHdm84cTcvQTVEaTVGT0RUOVVJUEprcjl6TzJiT3NUWDF5b29TRGZyYXpDMytOenpDdjVTc3FhN21ncnM4RkJGalhzRzJRbmhoWnlaZWJEZ2h2SHUzMEpEcUZQcmY3cFpOKy9tVFVDYnFqU200L0ZoU1JwMmFDeHlwRUxwMHVIWFVEZk9oYXFWL1MxbXNkVnkzSmZ6WU4xcHZxL1lrTE54dzBQdkpsQmNwdktOUS9CV0NMMlNnclpZS3RsaUVpMU1PSUdNV0d0R2tHeUkxNlRlR0U0eTVrWVZIdkZSZ2FkZ1MvbjNhWWEvR1d4SXZrcU5yRmIzUjdwN0tLUHgrZ3l4c2FxL0U0cHZrV0REYjFER1M1RG94bG5iazJ6aXpDVmRsUmJISEQ4cmhuNUwzUGtpTTJFTDNVL1J2KzdwU3lxL09JVDgxdmVoRDdXM2FaMGRWQmlNVjI0ZTlTNWdPZ3U5bm80WU8wNkFkeHVZbFdJT043ZG5URUdJVGttTStiL0I2Q0hVUWRYSXJTcDZLWW5wZTVVT2hhQW55RFovMHFYN29DQ3FFTWlVVDhZTDBEeVhaSmJtblEyR3Yxa3RJcTAwQklMYmJpL3Y3RnluUW9BZ1ZhSzFYbGFvRFNud0lIdVFLSjhEYzRvcUR5dDhWcG9oVWYxQVo2OXJDOWhpZnluL0xPL3MyT2NBcDYvY1VZVlY0UDJvNnZheGZrbWxNS0R3R1dyK2xMdWpxNkFOWFEvTTBCcTMwd1hKdUVtOW1Tc0xuYmNONWRWbTlDRi8zRXNDcTI4MTlZR2pabXRnQ1JXWnA5RWF2Y3ZHV0piVmo4T25UZ0prVVZHQ2R6aFlEdGRqYWdKenEvTW5NTnF2NVAzdUlGZVBoVUZZNnRVMEVib2lKVkRLUFNBK3dqVGgvQ3JBNWtPWlhXN01nWWUyUHBhK1ViRlZUUjlsN0J1ZDhCWjFtTDh2dXJqbXVMYmxzcG9mR0ZMUVp0Q0J5RzJFVXlnS0pVZ3hYc25Oa3hxY1BGMlhYdmxUSC9nbzFXYWd6WUV1RkViK3MxK0tncE9SUXhoR2duRFloeVBVLyt3QzBrc2E3bExtQUJuTUdVQ01HQThJZlpFdXM4eTdIdkxOaWN0cndFbnN4N2pEWmNJVlV6eWVzUWJ3cm

In [14]:
pretty_print(training_job)

{
  "training_job_name": "meta-textgeneration-llama-3-2-1b-instruct-rlaif-20260717005252",
  "training_job_arn": "arn:aws:sagemaker:us-west-2:674622101542:training-job/meta-textgeneration-llama-3-2-1b-instruct-rlaif-20260717005252",
  "model_artifacts": "s3_model_artifacts='s3://notebook-test-engine-ds-674622101542-usw2/output/meta-textgeneration-llama-3-2-1b-instruct-rlaif-20260717005252/output/model'",
  "training_job_status": "Completed",
  "secondary_status": "Completed",
  "hyper_parameters": {
    "clip_ratio": "0.2",
    "clip_ratio_high": "0.2",
    "clip_ratio_low": "0.2",
    "global_batch_size": "128",
    "judge_model_id": "bedrock/openai.gpt-oss-120b-1:0",
    "judge_prompt_template": "/opt/ml/input/data/judge_prompt/reward_prompt.jinja",
    "kl_loss_coef": "0.001",
    "learning_rate": "1e-05",
    "lora_alpha": "64",
    "lora_rank": "32",
    "lr_warmup_steps_ratio": "0.0",
    "max_epochs": "2",
    "max_prompt_length": "4000",
    "min_lr": "0.0",
    "model_name_or_